# Siteora Image Model — LoRA Training

Train a Siteora-specific LoRA adapter on top of a pretrained SDXL base model. Run cells top to bottom. Designed for a Colab T4 (16GB).

**Not tested on TPU v5e-1** — the code path exists (see `src/training/accelerator.py`) but only the CUDA/T4 and CPU-smoke-test paths have actually been run. See the repo README's 'Tested vs Expected' section.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt
# Colab ships a CUDA-enabled torch already; only add the extras we need on top of it.
!pip install -q diffusers==0.40.0 transformers==5.17.0 peft==0.21.0 accelerate==1.15.0 bitsandbytes>=0.43.0


## Cell 2 — Connect Google Drive (optional, recommended)

Used for checkpoint/LoRA persistence across Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/siteora-image-model'


## Cell 3 — Detect accelerator

In [ ]:
import sys
sys.path.insert(0, 'src')
from training.accelerator import detect_accelerator
accel_info = detect_accelerator()


## Cell 4 — Configure training

Pick one of `configs/test.yaml`, `configs/standard.yaml`, `configs/extended.yaml`, or point at your own. `test.yaml` is for verifying the pipeline; `standard.yaml` is the practical default.

In [ ]:
from training.config import SiteoraConfig

CONFIG_PATH = 'configs/test.yaml'  # change to configs/standard.yaml for a real run
config = SiteoraConfig.from_yaml(CONFIG_PATH)
config.paths.drive_root = DRIVE_ROOT if 'DRIVE_ROOT' in dir() else None
config.validate()
print(config)


## Cell 5 — Load / prepare dataset

See `colab/dataset.ipynb` for building `dataset/metadata.jsonl` from raw images. This cell assumes it already exists at `config.dataset.train_dir`.

In [ ]:
from dataset.preprocessing import dataset_statistics
print(dataset_statistics(config.dataset.train_dir, config.dataset.metadata_file))


## Cell 6 — Validate dataset

Training refuses to start on an invalid dataset (missing files, missing captions, corrupted images).

In [ ]:
from dataset.validation import require_valid_dataset
report = require_valid_dataset(config.dataset.train_dir, metadata_file=config.dataset.metadata_file)
print(report.summary())


## Cell 7 — Load base model

In [ ]:
from training.trainer import load_base_model
components = load_base_model(config, accel_info)


## Cell 8 — Attach LoRA

In [ ]:
from training.trainer import attach_lora, count_parameters
components.unet = attach_lora(components.unet, config)
trainable, total = count_parameters(components.unet)
print(f'Trainable: {trainable:,} / Total: {total:,} ({100*trainable/total:.3f}%)')


## Cell 9 — Run smoke test

Always run this before a long training run — verifies forward/backward/checkpoint/generate work on THIS environment before committing GPU-hours.

In [ ]:
!python tests/smoke_test.py


## Cell 10 — Start training

In [ ]:
from training.trainer import train
lora_dir = train(config, resume=True)
print('LoRA saved to', lora_dir)


## Cell 11 — View training metrics

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {config.paths.log_dir}


## Cell 12 — Generate evaluation images

In [ ]:
from evaluation.evaluate import run_evaluation
run_evaluation('final', 'evaluations', base_model=config.model.base_model, lora_path=str(lora_dir))


## Cell 13 — Save / export LoRA

`lora_dir` already contains the exportable adapter (see `models/<name>/` — a small adapter, not the full base model). Copy it to Drive if not already synced.

In [ ]:
from training.checkpointing import sync_to_drive
sync_to_drive(lora_dir, config.paths.drive_root)
